In [ ]:
# Generate a 1000-line realistic app.log for Session 2

import random
from datetime import datetime, timedelta

output_path = "process.log"

levels = ["INFO", "WARNING", "ERROR", "DEBUG"]
endpoints = ["/", "/login", "/logout", "/dashboard", "/api/data", "/api/pay", "/profile", "/settings"]
messages = {
    "INFO": ["User authenticated", "Request completed", "Cache refreshed", "Service started"],
    "WARNING": ["High memory usage", "Slow response detected", "Disk space low"],
    "ERROR": ["Payment failed", "Database timeout", "Unhandled exception", "Authentication failed"],
    "DEBUG": ["Variable x initialized", "Entering function processData()", "Payload received"]
}

start_time = datetime(2025, 2, 14, 8, 0, 0)

with open(output_path, "w") as f:
    for i in range(1000):
        timestamp = start_time + timedelta(seconds=i * random.randint(1, 5))
        level = random.choice(levels)
        ip = f"{random.randint(10, 192)}.{random.randint(0,255)}.{random.randint(0,255)}.{random.randint(1,254)}"
        endpoint = random.choice(endpoints)
        status = random.choice([200, 200, 200, 404, 500, 503])  # more 200s for realism
        message = random.choice(messages[level])
        
        line = f"{timestamp} {level} {ip} \"GET {endpoint}\" {status} - {message}\n"
        f.write(line)

output_path


FileNotFoundError: [Errno 2] No such file or directory: '/mnt/data/process.log'

In [ ]:
import random
import re
from datetime import datetime, timedelta

APP_LOG_PATH = "app.log"
PROCESS_LOG_PATH = "process.log"

# Services/processes to simulate
PROCESSES = [
    "api-service",
    "auth-service",
    "payment-service",
    "worker",
    "db-proxy",
    "cache",
]

STATUSES = ["RUNNING", "RUNNING", "RUNNING", "DEGRADED", "RESTARTING"]  # weighted toward RUNNING

def parse_app_log_timestamps(app_path: str, max_lines: int = 1000):
    """
    Attempts to parse timestamps from app.log lines that start like:
    2026-02-14 08:00:00 ...
    Returns a list of datetime objects if possible; otherwise returns [].
    """
    ts = []
    with open(app_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= max_lines:
                break
            # Match "YYYY-MM-DD HH:MM:SS"
            m = re.match(r"^(\d{4}-\d{2}-\d{2})\s+(\d{2}:\d{2}:\d{2})", line)
            if m:
                dt = datetime.strptime(m.group(1) + " " + m.group(2), "%Y-%m-%d %H:%M:%S")
                ts.append(dt)
    return ts

def generate_process_log(out_path: str, timestamps=None, lines: int = 1000):
    """
    If timestamps is provided, uses them (cycling if shorter than requested).
    Otherwise generates its own timeline.
    """
    if not timestamps:
        start = datetime(2026, 2, 14, 8, 0, 0)
        timestamps = [start + timedelta(seconds=i*2) for i in range(lines)]

    with open(out_path, "w", encoding="utf-8") as f:
        for i in range(lines):
            dt = timestamps[i % len(timestamps)]

            proc = random.choice(PROCESSES)

            # Baselines per process (rough realism)
            base_cpu = {
                "api-service": 25,
                "auth-service": 15,
                "payment-service": 20,
                "worker": 35,
                "db-proxy": 18,
                "cache": 10,
            }[proc]

            base_mem = {
                "api-service": 180,
                "auth-service": 140,
                "payment-service": 200,
                "worker": 260,
                "db-proxy": 160,
                "cache": 120,
            }[proc]

            # Random variation
            cpu = max(0, min(100, int(random.gauss(base_cpu, 10))))
            mem = max(50, int(random.gauss(base_mem, 30)))

            # Inject occasional spikes (for awk numeric filtering exercises)
            if random.random() < 0.06:
                cpu = random.randint(80, 98)  # spike
                mem += random.randint(80, 200)

            status = random.choice(STATUSES)

            # Make spikes more likely to coincide with non-RUNNING statuses
            if cpu >= 85 and random.random() < 0.5:
                status = random.choice(["DEGRADED", "RESTARTING"])

            line = (
                f"{dt:%Y-%m-%d %H:%M:%S} PROCESS {proc} "
                f"CPU={cpu}% MEM={mem}MB STATUS={status}\n"
            )
            f.write(line)

if __name__ == "__main__":
    # Option A: sync with app.log timestamps (recommended for cross-file exercises)
    timestamps = parse_app_log_timestamps(APP_LOG_PATH)

    # If parsing failed, it will generate its own timeline
    generate_process_log(PROCESS_LOG_PATH, timestamps=timestamps, lines=1000)

    print(f"Generated {PROCESS_LOG_PATH} (1000 lines).")